In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from math import sqrt

# Input and Output folders
input_folder = "../5_Weather_Merged_CSVs/Test"
output_folder = "../7_Results/Sarimax_Results"
os.makedirs(output_folder, exist_ok=True)

# Store results
results = []

# Loop through all CSV files
for file in os.listdir(input_folder):
    if file.endswith(".csv"):
        file_path = os.path.join(input_folder, file)
        crop_name = os.path.splitext(file)[0]  # Filename without extension

        try:
            # Load Data
            df = pd.read_csv(file_path)
            df['Price Date'] = pd.to_datetime(df['Price Date'], errors='coerce')
            df = df.sort_values('Price Date')
            df.set_index('Price Date', inplace=True)

            # Drop NA rows if any
            df = df.dropna()

            # Define target + features
            y = df['Modal Price (Rs./Quintal)']
            X = df[['Day Of Week', 'lookback_temp_mean', 'lookback_precip_sum']]

            # Train/Test Split
            train_size = int(len(y) * 0.8)
            y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]
            X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]

            # Fit SARIMAX
            model = SARIMAX(y_train, exog=X_train, order=(1,1,1), seasonal_order=(1,1,1,7))
            model_fit = model.fit(disp=False)

            # Forecast
            y_pred = model_fit.predict(start=len(y_train), end=len(y)-1, exog=X_test)

            # Metrics
            rmse = sqrt(mean_squared_error(y_test, y_pred))
            nrmse = rmse / (y_test.max() - y_test.min())
            mape = mean_absolute_percentage_error(y_test, y_pred) * 100
            accuracy = 100 - mape

            # Save metrics
            results.append([crop_name, rmse, nrmse, mape, accuracy])

            # Plot
            plt.figure(figsize=(12,6))
            plt.plot(y, label="Actual", color="red")
            plt.plot(y_test.index, y_pred, label="Predicted", color='blue')
            plt.title(f"{crop_name} - SARIMAX Forecast (MAPE: {mape:.2f}%)")
            plt.legend()
            plt.savefig(os.path.join(output_folder, f"{crop_name}_forecast.png"))
            plt.close()

            print(f"✅ Done: {crop_name}")

        except Exception as e:
            print(f"❌ Failed for {crop_name}: {e}")

# Save all results into CSV
results_df = pd.DataFrame(results, columns=["Crop", "RMSE", "NRMSE", "Loss% (MAPE)", "Accuracy"])
results_df.to_csv(os.path.join(output_folder, "Sarima_Results.csv"), index=False)

print("\n📊 All results saved in Sarima_Results folder!")


c:\Users\siddh\OneDrive\Documents\Agricultural Datasets\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\siddh\OneDrive\Documents\Agricultural Datasets\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\siddh\OneDrive\Documents\Agricultural Datasets\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\siddh\OneDrive\Documents\Agricultural Datasets\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is ava

✅ Done: Mango-Raw-Ripe

📊 All results saved in Sarima_Results folder!
